# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.functions import regexp_replace, col, initcap, lower, trim, when, lit, StringType, count, isnan, upper
from pyspark.sql.types import DecimalType, BooleanType
from functools import reduce

# Criando o database silver

In [0]:
%sql
USE CATALOG projeto;
DROP DATABASE silver CASCADE; -- Executar se tiver uma silver já criada
CREATE DATABASE IF NOT EXISTS silver;

## Tabela: Atendentes

In [0]:
# # CÉLULA: TRANSFORMAÇÃO - base_atendentes para Silver (CORRIGIDO)

# # Lendo a tabela bronze
# df_atendentes_bronze = spark.table("projeto.bronze.base_atendentes")
# print(f"📊 Registros na bronze: {df_atendentes_bronze.count()}")

# # Mostrar schema original
# print("📋 SCHEMA ORIGINAL:")
# df_atendentes_bronze.printSchema()

# # TRANSFORMAÇÕES

# df_atendentes_silver = (
#     df_atendentes_bronze
#     .dropDuplicates(["id_atendente"])
#     .withColumn("nome_atendente", 
#                 F.initcap(F.trim(F.col("nome_atendente"))))
#     .withColumn("nivel_atendimento", 
#                 F.col("nivel_atendimento").cast("int"))
#     .withColumn("descricao_nivel",
#                 F.when(F.col("nivel_atendimento") == 1, "Nível 1 - Suporte Básico")
#                  .when(F.col("nivel_atendimento") == 2, "Nível 2 - Suporte Especializado")
#                  .when(F.col("nivel_atendimento") == 3, "Nível 3 - Suporte Avançado")
#                  .otherwise("Nível Não Especificado"))
#     .withColumn("id_atendente", F.col("id_atendente").cast("int"))
#     .withColumn("nome_atendente", F.col("nome_atendente").cast("string"))
#     .withColumn("nivel_atendimento", F.col("nivel_atendimento").cast("int"))
#     .withColumn("descricao_nivel", F.col("descricao_nivel").cast("string"))
#     .withColumn("silver_ingestion_timestamp", F.current_timestamp())
#     .withColumn("source_file", F.col("source_file"))

#     .select(
#         "id_atendente",
#         "nome_atendente", 
#         "nivel_atendimento",
#         "descricao_nivel",
#         "silver_ingestion_timestamp",
#         "source_file"
#     )
# )

# df_atendentes_silver.printSchema()
# display(df_atendentes_silver)

In [0]:
# # VERIFICAÇÃO DOS VALORES PADRONIZADOS

# print("VERIFICAÇÃO DOS VALORES PADRONIZADOS")

# print("\nDISTRIBUIÇÃO POR NÍVEL DE ATENDIMENTO:")
# distribuicao_nivel = df_atendentes_silver.groupBy("nivel_atendimento", "descricao_nivel").count().orderBy("nivel_atendimento")
# display(distribuicao_nivel)

# print("\nNOMES DE ATENDENTES (amostra):")
# nomes_amostra = df_atendentes_silver.select("nome_atendente").collect()
# for nome in nomes_amostra:
#     print(f"  - {nome['nome_atendente']}")

# print(f"\nTOTAL DE ATENDENTES POR NÍVEL:")
# for row in distribuicao_nivel.collect():
#     print(f"  - {row['descricao_nivel']}: {row['count']} atendentes")

# print(f"\nATENDENTES DO NÍVEL 2 (Especializados):")
# nivel_2 = df_atendentes_silver.filter(F.col("nivel_atendimento") == 2).select("id_atendente", "nome_atendente")
# display(nivel_2)

In [0]:
# # SALVANDO NA CAMADA SILVER

# try:
#     spark.sql("CREATE SCHEMA IF NOT EXISTS projeto.silver")
# except Exception as e:
#     print(f"Erro ao criar schema: {e}")

# # Salvar a tabela
# try:
#     (
#         df_atendentes_silver
#         .write
#         .format("delta")
#         .mode("overwrite")
#         .option("overwriteSchema", "true")
#         .saveAsTable("projeto.silver.dim_atendentes")
#     )
#     print("✅ TABELA SALVA COM SUCESSO!")
#     print("📊 Tabela: projeto.silver.dim_atendentes")
    
# except Exception as e:
#     print(f"❌ Erro ao salvar tabela: {e}")

# # Verificação final
# try:
#     df_verificacao = spark.table("projeto.silver.dim_atendentes")
#     print(f"✅ Registros confirmados na silver: {df_verificacao.count()}")
    
#     print("\n📋 SCHEMA DA TABELA SALVA:")
#     df_verificacao.printSchema()
    
# except Exception as e:
#     print(f"❌ Erro na verificação: {e}")

## Tabela: Canais

Para a tabela de dm.canais, é preciso garantir que os tipos de dados bem como os nomes das colunas estejam corretos conforme o grupo definiu.

Schema final da tabela:

```
root
 |-- Nome_Canal: string (nullable = false)
 |-- Status_Canal: string (nullable = false)
```

In [0]:
# criação da sessão do Spark
spark = SparkSession.builder.appName("bronze_to_silver").getOrCreate()

# paths dos schemas
bronze_path = "bronze"
silver_path = "silver"

canais = spark.read.table(f"{bronze_path}.canais")

# checando o schema atual dos dados da bronze layer
canais.printSchema()

In [0]:
canais.display() # checando os dados

De acordo com o observado acima, a coluna ```nome_canal``` possui uma despadronização quanto à forma que as palavras estão escritas. Para padronizar, vamos capitalizar todas elas e caso tenha alguma ocorrência futura que seja nula, vamos colocar o valor de ```Desconhecido```.

Já na coluna de ```canal_status```, existe uma padronização quanto às diferentes escritas de inativo (sejam corretas no vocabulário português ou não). Para padronizar, vamos checar a primeira letra de cada ocorrência e atribuir à um valor constante, que será ```Ativo```, ```Inativo``` ou ```Desconhecido```, caso o valor da coluna seja nulo.

In [0]:
canais = (
    canais
    # nome_canal
    .withColumn("nome_canal", 
                # se o nome do canal for nulo, substitui por desconhecido
                F.coalesce(upper(col("nome_canal")), lit("desconhecido"))
    )
    .withColumn("nome_Canal", col("nome_Canal").cast(StringType()))
    
    # canal_status
    .withColumn("canal_status", 
                when(upper(col("canal_status")).startswith("A"), True)
                .otherwise(False)
                .cast(BooleanType())
    )
)

In [0]:
# checando alterações pós tratamento
canais.display()

In [0]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

Todas as mudanças foram efetivas e deixou a coluna padronizada para o futuro.

Como existem somente 6 ocorrências dos dados, não é possível criar futuras Views somente com esta tabela, somente em conjunto de outras tabelas.

Com isso, resta partir para o salvamento da tabela na Silver Layer.

In [0]:
canais.write.format("delta").mode("overwrite").saveAsTable(f"{silver_path}.dim_canais")

## Tabela: Chamados_Hora

In [0]:
# Lendo a tabela chamados_hora da camada bronze e visualizando os primeiros registros
df_bz = spark.table("bronze.chamados_hora")
print(f"bronze.chamados_hora: {df_bz.count()} rows")
display(df_bz.limit(10))

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# 1) Remoção de duplicatas
df_max = (
    df_bz.groupBy("ID_Chamado")
         .agg(F.max("ingestion_timestamp").alias("ingestion_timestamp"))
)

df_bz = df_bz.join(
    df_max,
    on=["ID_Chamado", "ingestion_timestamp"],
    how="inner"
)

# 2) Normalização de colunas + timestamps + métricas
def to_ts(col):
    return F.to_timestamp(F.regexp_replace(col, " �s ", " "), "dd/MM/yyyy HH:mm:ss")

df = (
    df_bz
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado_raw")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento_raw")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento_raw")
    .withColumn("data_hora_abertura", to_ts(F.col("hora_abertura_chamado_raw")))
    .withColumn("data_hora_inicio_atendimento", to_ts(F.col("hora_inicio_atendimento_raw")))
    .withColumn("data_hora_finalizacao_atendimento", to_ts(F.col("hora_finalizacao_atendimento_raw")))
    .drop(
        "hora_abertura_chamado_raw",
        "hora_inicio_atendimento_raw",
        "hora_finalizacao_atendimento_raw"
    )
    .withColumn("id_chamado", F.col("id_chamado").cast("long"))
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))
    .withColumn(
        "tempo_espera_atendimento_min",
        F.round(
            (F.col("data_hora_inicio_atendimento").cast("long") - F.col("data_hora_abertura").cast("long")) / 60.0,
            2
        )
    )
    .withColumn(
        "tempo_atendimento_min",
        F.round(
            (F.col("data_hora_finalizacao_atendimento").cast("long") - F.col("data_hora_inicio_atendimento").cast("long")) / 60.0,
            2
        )
    )
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))
)

# 3) Dimensão calendário
range_datas = df.select(
    F.min(F.to_date("data_hora_abertura")).alias("min_data"),
    F.max(F.to_date("data_hora_abertura")).alias("max_data")
).first()

min_date = range_datas["min_data"].strftime("%Y-%m-%d")
max_date = range_datas["max_data"].strftime("%Y-%m-%d")

df_datas = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{min_date}'),
        to_date('{max_date}'),
        interval 1 day
    )) AS data_referencia
""")

df_calendario = (
    df_datas
    .withColumn("ano", F.year("data_referencia"))
    .withColumn("mes", F.month("data_referencia"))
    .withColumn("dia", F.dayofmonth("data_referencia"))
    .withColumn("dia_semana_num", F.dayofweek("data_referencia"))
    .withColumn("nome_dia_en", F.date_format("data_referencia", "EEEE"))
    .withColumn(
        "nome_dia",
        F.when(F.col("nome_dia_en") == "Monday", "SEGUNDA-FEIRA")
         .when(F.col("nome_dia_en") == "Tuesday", "TERCA-FEIRA")
         .when(F.col("nome_dia_en") == "Wednesday", "QUARTA-FEIRA")
         .when(F.col("nome_dia_en") == "Thursday", "QUINTA-FEIRA")
         .when(F.col("nome_dia_en") == "Friday", "SEXTA-FEIRA")
         .when(F.col("nome_dia_en") == "Saturday", "SABADO")
         .when(F.col("nome_dia_en") == "Sunday", "DOMINGO")
         .otherwise(None)
    )
    .drop("nome_dia_en")
    .withColumn("trimestre", F.quarter("data_referencia"))
    .withColumn("semana_do_ano", F.weekofyear("data_referencia"))
    .withColumn(
        "flag_fim_de_semana",
        F.col("dia_semana_num").isin(1, 7)  # bool
    )
    .withColumn(
        "ciclo_operacional",
        F.when(F.col("dia") <= 10, "INICIO MES")
         .when(F.col("dia") >= 25, "FIM MES")
         .otherwise("MEIO MES")
    )
    .withColumn(
        "evento_sazonal",
        F.when(F.col("data_referencia") == "2025-01-01", "ANO NOVO")
         .when(F.col("data_referencia").between("2025-02-28", "2025-03-05"), "CARNAVAL")
         .when(F.col("data_referencia") == "2025-04-18", "SEXTA-FEIRA SANTA")
         .when(F.col("data_referencia") == "2025-04-21", "TIRADENTES")
         .when(F.col("data_referencia") == "2025-05-01", "DIA DO TRABALHO")
         .when(F.col("data_referencia") == "2025-06-19", "CORPUS CHRISTI")
         .when(F.col("data_referencia") == "2025-09-07", "INDEPENDENCIA")
         .when(F.col("data_referencia") == "2025-10-12", "NOSSA SENHORA APARECIDA")
         .when(F.col("data_referencia") == "2025-11-02", "FINADOS")
         .when(F.col("data_referencia") == "2025-11-15", "PROCLAMACAO DA REPUBLICA")
         .when(F.col("data_referencia") == "2025-12-25", "NATAL")
         .otherwise("DIA COMUM")
    )
)

# 4) Join fato-hora + calendário
condicao_join = (
    F.to_date(df["data_hora_abertura"]) == df_calendario["data_referencia"]
)

df_final = (
    df.join(df_calendario, on=condicao_join, how="left")
      .drop("data_referencia")
)

df_final = df_final.select(
    "id_chamado",
    "id_cliente",
    "data_hora_abertura",
    "data_hora_inicio_atendimento",
    "data_hora_finalizacao_atendimento",
    "tempo_espera_atendimento_min",
    "tempo_atendimento_min",
    "ano",
    "mes",
    "dia",
    "dia_semana_num",
    "nome_dia",
    "trimestre",
    "semana_do_ano",
    "flag_fim_de_semana",
    "ciclo_operacional",
    "evento_sazonal",
    "ingestion_timestamp"
)

df_final.write.mode("overwrite").saveAsTable("silver.dim_chamados_data")

print(f"silver.dim_chamados_data: {df_final.count()} rows")
display(df_final.limit(10))


## Tabela: Clientes

In [0]:
# Lendo a tabela de clientes da camada Bronze
df_clientes_bronze = spark.table("bronze.clientes")

print(f"Total de registros na bronze: {df_clientes_bronze.count()}")
display(df_clientes_bronze.limit(10))

In [0]:
def analyze_missing_data(df_origem):
    # 1. Definir explicitamente as colunas alvo
    colunas_alvo = ["id_cliente", "nome", "email", "regiao", "idade"]
    
    print(f"Analisando apenas as colunas: {colunas_alvo}")
    
    # Seleciona apenas estas colunas para evitar erros com timestamps ou colunas extras
    df_check = df_origem.select(*colunas_alvo)
    total_rows = df_check.count()

    # 2. Preparar condições (seguro contra erro de tipo)
    expressoes_agg = []
    condicoes_filtro = []
    
    for c_name, c_type in df_check.dtypes:
        # isNaN só funciona em float/double. Para string/int usamos apenas isNull
        if c_type in ('double', 'float'):
            condicao = col(c_name).isNull() | isnan(c_name)
        else:
            condicao = col(c_name).isNull()
            
        expressoes_agg.append(count(when(condicao, c_name)).alias(c_name))
        condicoes_filtro.append(condicao)
    
    # 3. Exibir Resumo (Contagem e Porcentagem)
    resultado_agg = df_check.select(expressoes_agg).collect()[0].asDict()
    
    dados_resumo = []
    for coluna, qtd_missing in resultado_agg.items():
        dados_resumo.append({
            "Coluna": coluna,
            "Faltantes": qtd_missing,
            "%": round((qtd_missing / total_rows) * 100, 2)
        })
        
    print("--- Resumo de Faltantes ---")
    display(spark.createDataFrame(dados_resumo).orderBy(col("Faltantes").desc()))
    
    # 4. Mostrar linhas que possuem QUALQUER um dos campos acima nulo
    if condicoes_filtro:
        condicao_final = reduce(lambda x, y: x | y, condicoes_filtro)
        linhas_com_problema = df_check.filter(condicao_final)
        
        if linhas_com_problema.count() > 0:
            print("--- Amostra de linhas com dados faltando ---")
            display(linhas_com_problema.limit(10))
        else:
            print("Sucesso! Nenhuma linha com dados faltantes nessas colunas.")

# Executando a análise
analyze_missing_data(df_clientes_bronze)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as W

# Reiniciando a referência apenas por segurança
df_clientes = spark.table("bronze.clientes")

# 1. Tratamento de Região (Remover acentos e deixar maiúsculo)
acentos     = "áéíóúàèìòùâêîôûãõçÁÉÍÓÚÀÈÌÒÙÂÊÎÔÛÃÕÇ"
sem_acentos = "aeiouaeiouaeiouaocAEIOUAEIOUAEIOUAOC"

df_clientes = df_clientes.withColumn(
    "regiao_clean", 
    F.upper(F.translate(F.col("regiao"), acentos, sem_acentos))
)

# 2. Validação de E-mail e Tipagem
# Regex atualizado para aceitar o caractere '+' (ex: elis.teixeira+5037@...)
email_pattern = r"^[\w\.\-\+]+@[\w\.-]+\.\w+$"

df_clientes = (
    df_clientes
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))
    .withColumn("idade", F.col("idade").cast("integer"))
    
    # Filtra mantendo apenas e-mails válidos
    .filter(F.col("email").rlike(email_pattern))
    
    # Troca a coluna de região antiga pela tratada
    .drop("regiao")
    .withColumnRenamed("regiao_clean", "regiao")
)

# 3. Deduplicação (Mantém o registro mais recente baseado na ingestão)
window_spec = W.partitionBy("id_cliente").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = (
    df_clientes
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# 4. Seleção Final
df_final = df_dedup.select(
    "id_cliente",
    "nome",
    "email",
    "regiao",
    "idade",
    "ingestion_timestamp"
)

In [0]:
# Salvando na tabela Silver (formato Delta)
df_final.write.mode("overwrite").saveAsTable("silver.dim_clientes")

print(f"Linhas finais salvas na silver.dim_clientes: {df_final.count()}")
display(df_final.limit(10))

In [0]:
df_silver_dim_clientes_geracoes = df_final \
    .withColumn(
        "faixa_etaria_geracao",
        F.when(F.col("idade") < 28, "Gen Z") ##N FIZ OUTRA GERAÇÃO, POIS É UMA LOJA DE FINANCIAMENTO ENT N TERÃO IDADES MENORES QUE 18
         .when((F.col("idade") >= 28) & (F.col("idade") < 44), "Millennials")
         .when((F.col("idade") >= 44) & (F.col("idade") < 60), "Gen X")
         .otherwise("Boomers") 
    ) \
    .withColumn(
        "flag_idoso",
        F.when(F.col("idade") >= 60, 1).otherwise(0)
    ) \

df_silver_dim_clientes_geracoes = df_silver_dim_clientes_geracoes.select(
    "id_cliente",
    "nome",
    "email",
    "regiao",
    "idade",
    "faixa_etaria_geracao",
    "flag_idoso",
)

display(df_silver_dim_clientes_geracoes)


df_silver_dim_clientes_geracoes.write.mode("overwrite").saveAsTable("projeto.silver.fato_clientes_geracoes")



## Tabela: Custos

In [0]:
# Lendo a tabela e vizualizando os primeiros registros
df_custos_bronze = spark.table("bronze.custos")
print(f"{df_custos_bronze.count()} rows")
display(df_custos_bronze.limit(10))

In [0]:
# Removendo duplicatas
df_custos_bronze = df_custos_bronze.distinct()

# Limpando os dados, manténdo apenas números, vírgula e ponto
df_clean = df_custos_bronze.withColumn(
    "custo_limpo",
    regexp_replace(col("custo"), "[^0-9,\\.]", "") 
)

# Trocando vírgula por pontos 
df_clean = df_clean.withColumn(
    "custo_padronizado",
    regexp_replace(col("custo_limpo"), ",", ".")
)


# Alterando o tipo de dados de custo para decimal
df_clean = df_clean.withColumn(
    "custo_final",
    col("custo_padronizado").cast(DecimalType(18, 10))
)

# Padronizando nome das colunas com snake_case
df_clean = df_clean.withColumnRenamed("custo_final", "valor_custo")

df_custos_silver = df_clean.select(
    "id_custo",
    "id_chamado",
    "valor_custo"
)

# Atualizando tempo de ingenstão para camanda silver
df_custos_silver = df_custos_silver.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
df_custos_silver.write.mode("overwrite").saveAsTable("silver.dim_custos")

In [0]:
print(f"{df_custos_silver.count()} rows")
display(df_custos_silver.limit(10))

## Tabela: Motivos

In [0]:
# Lendo e vizualizando dataset 
df_motivos_bronze = spark.table("bronze.base_motivos")
print(f"{df_motivos_bronze.count()} rows")
display(df_motivos_bronze.limit(10))

In [0]:
df_motivos_bronze = spark.table("bronze.base_motivos")
# Removendo duplicadas 
df_motivos_bronze = df_motivos_bronze.distinct()

# Corrigindo erros de digitação no -> não
df_motivos_bronze = df_motivos_bronze.withColumn(
    "nome_motivo",
    F.when(F.col("nome_motivo") == "Compra no autorizada", "Compra não autorizada")
     .otherwise(F.col("nome_motivo"))
)

df_motivos_bronze = df_motivos_bronze.withColumn(
    "nome_motivo_clean",
    regexp_replace(
        col("nome_motivo"),
        "[áàâãäÁÀÂÃÄéèêëÉÈÊËíìîïÍÌÎÏóòôõöÓÒÔÕÖúùûüÚÙÛÜçÇ]",
        ""
    )
)

# Colocando os valores em upper
df_motivos_bronze = df_motivos_bronze.withColumn("nome_motivo_clean", upper(col("nome_motivo_clean")))

df_motivos_bronze = df_motivos_bronze.withColumn("nome_motivo",
    F.translate(
        col("nome_motivo"),
        "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
        "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"
    )
)

df_motivos_bronze = df_motivos_bronze.withColumn("criticidade",
    F.translate(
        col("criticidade"),
        "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
        "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"
    )
)

# Se baseando nos crietrios de categorização e nos motivos mais comuns entre as demandas de atendimento essa perta atribui as categorias de cada motivo atraves de Keywords presentes no seu nome do motivo
df_motivos_bronze = df_motivos_bronze.withColumn(
            "categoria",
            when(col("nome_motivo").rlike("(?i)fatura|limite|contrato|divida|renegociacao"), "Financeiro")
            .when(col("nome_motivo").rlike("(?i)cart[aã]o|compra"), "Cartao")
            .when(col("nome_motivo").rlike("(?i)dados|telefone|email|agencia|vencimento|cancel|encerr|fechamento"), "Cadastral")
            .when(col("nome_motivo").rlike("(?i)aplicativo|app|site|chatbot|ura"), "Atendimento")
            .when(col("nome_motivo").rlike("(?i)pontos|benef"), "Beneficios")
            .otherwise(None)
        )

# Colocando os valores em upper
df_motivos_bronze = df_motivos_bronze.withColumn("nome_motivo", upper(col("nome_motivo")))

# Padronizando formato de escrita coluna de criticidade
df_motivos_bronze = df_motivos_bronze.withColumn("criticidade", initcap(lower(trim(col("criticidade")))))

# Removendo espaços extra no nome
df_motivos_bronze = df_motivos_bronze.withColumn("nome_motivo", trim(col("nome_motivo")))

df_motivos_silver = df_motivos_bronze.withColumn("ingestion_timestamp", current_timestamp())


# Renomeando colunas para seguir estilo snake_case
df_motivos_silver = (
    df_motivos_silver
    .withColumnRenamed("categoria", "categoria_motivo")
    .withColumnRenamed("criticidade", "criticidade_motivo")
)


     

In [0]:
df_motivos_silver.write.mode("overwrite").saveAsTable("silver.dim_motivos")

In [0]:
print(f"{df_motivos_silver.count()} rows")
display(df_motivos_silver.limit(10))

## Tabela: Pesquisa_Satisfação

In [0]:
df_bronze_satisfacao = spark.table("projeto.bronze.pesquisa_satisfacao") 

total_linhas = df_bronze_satisfacao.count()
total_chamados_unicos = df_bronze_satisfacao.select("id_chamado").distinct().count()
total_pesquisas_unicas = df_bronze_satisfacao.select("id_pesquisa").distinct().count()
qtd_duplicatas_chamado = total_linhas - total_chamados_unicos

qtd_nulos_nota = df_bronze_satisfacao.filter(F.col("nota_atendimento").isNull()).count()

df_bronze_satisfacao_outliers = df_bronze_satisfacao.filter(
    (F.col("nota_atendimento") < 1) | 
    (F.col("nota_atendimento") > 5)
)
qtd_outliers = df_bronze_satisfacao_outliers.count()
df_bronze_satisfacao_dados_incompletos = df_bronze_satisfacao.filter(
    F.col("id_chamado").isNull() |
    F.col("id_pesquisa").isNull() |
    F.col("nota_atendimento").isNull() |
    F.col("ingestion_timestamp").isNull()
)

print("RESUMO DE QUALIDADE DE DADOS")
print(f"Total de Linhas:{total_linhas}")
print(f"Chamados Únicos:{total_chamados_unicos}")
print(f"IDs Pesquisa Únicos:{total_pesquisas_unicas}")
print(f"Duplicatas de Chamado:{qtd_duplicatas_chamado}")
print(f"Notas Nulas:{qtd_nulos_nota}")
print(f"Notas Outliers (<1 ou >5):{qtd_outliers}")
print(f"Registros Incompleto{df_bronze_satisfacao_dados_incompletos.count()}")

if df_bronze_satisfacao_dados_incompletos.count() > 0:
    print("Visualizando amostra de dados incompletos:")
    display(df_bronze_satisfacao_dados_incompletos.limit(5))

if qtd_outliers > 0:
    print("Visualizando amostra de outliers:")
    display(df_bronze_satisfacao_outliers.limit(5))
     



PERCEBO QUE EXISTEM MUITOS VALORES DE NOTA_ATENDIMENTO FALTANTE (FAZ SENTIDO POIS OS CLIENTES NÃO SÃO OBRIGADOS A ATRIBUIR UMA NOTA AO FUNCIONÁRO), MAS OS CAMPOS DE ID ESTÃO SEMPRE COMO ÚNICOS POR ENQUANTO, MAS PRECISO GARANTIR NO MEU JOB QUE ESSAS LINHAS NÃO TERÃO DUPLICATAS

AGORA IREI FAZER UM CAST NOS MEUS DADOS


In [0]:
df_bronze_satisfacao = (
    df_bronze_satisfacao
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    .withColumn("id_pesquisa", F.col("id_pesquisa").cast("int"))
    .withColumn("nota_atendimento", F.col("nota_atendimento").cast("int"))
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))
)

In [0]:
df_bronze_satisfacao.write.mode("overwrite").saveAsTable("projeto.silver.dim_pesquisa_satisfacao")

# Tabela: Fatos Chamados

In [0]:
df_bz = spark.table("bronze.chamados")
print(f"bronze.chamados: {df_bz.count()} rows")
display(df_bz.limit(10))

In [0]:
df_bz = spark.table("bronze.chamados")
# Limpeza + Conversão da coluna "resolvido" → boolean
df_clean = df_bz.withColumn(
    "resolvido_bool",
    when(upper(trim(col("resolvido"))).isin("SIM", "S"), True)
    .when(upper(trim(col("resolvido"))).isin("NAO", "NÃO", "N", "N�O"), False)
    .otherwise(None)
    .cast(BooleanType())
)
df_clean = df_clean.drop("resolvido").withColumnRenamed("resolvido_bool", "resolvido")

# JOIN entre chamados (bronze) e silver.dim_chamados_hora
df_dim_chamados_data = spark.table("silver.dim_chamados_data")
df_dim_pesquisa_satisfacao = spark.table("silver.dim_pesquisa_satisfacao")
df_dim_custos = spark.table("silver.dim_custos")
df_dim_motivos= spark.table("silver.dim_motivos")


# Processamento do nome do motivo
# Corrige os caracteres quebrados e remove acentos
# Processamento do nome do motivo
df_clean = df_clean.withColumn(
    "nome_motivo_clean",
    upper(
        regexp_replace(
            col("motivo"),
            "[áàâãäÁÀÂÃÄéèêëÉÈÊËíìîïÍÌÎÏóòôõöÓÒÔÕÖúùûüÚÙÛÜçÇ�]",
            ""
        )
    )
)


df_joined = (
    df_clean.alias("bz_chamados")
    .join(
        df_dim_chamados_data.alias("data"),
        col("bz_chamados.id_chamado") == col("data.id_chamado"),
        "inner"
    )

    .join(
        df_dim_pesquisa_satisfacao.alias("pesquisa"),
        col("bz_chamados.id_chamado") == col("pesquisa.id_chamado"),
        "left"
    )

    .join(
        df_dim_custos.alias("custos"),
        col("bz_chamados.id_chamado") == col("custos.id_chamado"),
        "left"
    )

    .join(
        df_dim_motivos.alias("motivos"),
        col("bz_chamados.nome_motivo_clean") == col("motivos.nome_motivo_clean"),
        "inner"
    )

    .select(
        col("bz_chamados.id_chamado"),
        col("bz_chamados.id_cliente"),
        col("data.data_hora_abertura"),
        col("data.data_hora_inicio_atendimento"),
        col("data.data_hora_finalizacao_atendimento"),
        col("data.tempo_espera_atendimento_min"),
        col("data.tempo_atendimento_min"),
        col("data.dia"),
        col("data.mes"),
        col("data.ano"),
        col("data.dia_semana_num"),
        col("data.nome_dia"),
        col("data.trimestre"),
        col("data.semana_do_ano"),
        col("data.flag_fim_de_semana"),
        col("data.ciclo_operacional"),
        col("data.evento_sazonal"),
        col("motivos.nome_motivo").alias("motivo"),
        col("bz_chamados.resolvido"),
        upper(col("bz_chamados.canal")).alias("nome_canal"),
        col("bz_chamados.ingestion_timestamp"),
        col("pesquisa.nota_atendimento"),
        col("custos.valor_custo")
    )
)

# Processamento do nome do canal
df_silver_fato_chamados = df_joined.withColumn(
        "nome_canal",
        upper(
            when(col("nome_canal").contains("ESP"), "ATENDIMENTO ESPECIALIZADO")
            .when(col("nome_canal").contains("INI"), "ATENDIMENTO INICIAL")
            .when((upper(col("nome_canal")).contains("CHAT")) | (upper(col("nome_canal")).contains("BOT")), "CHATBOT")
            .when(upper(col("nome_canal")).rlike("W.*E.*B"),"WEB")
            .when(upper(col("nome_canal")).rlike("U.*R.*A"),"URA")
            .when(upper(col("nome_canal")).rlike("E.*M.*A.*I.*L"), "EMAIL")
            .otherwise("DESCONHECIDO")
        )
    )

# Tipagem
df_silver_fato_chamados = (
    df_silver_fato_chamados
        .withColumn("id_chamado", col("id_chamado").cast("long"))
        .withColumn("id_cliente", col("id_cliente").cast("long"))
        .withColumn("data_hora_abertura", col("data_hora_abertura").cast("timestamp"))
        .withColumn("data_hora_inicio_atendimento", col("data_hora_inicio_atendimento").cast("timestamp"))
        .withColumn("data_hora_finalizacao_atendimento", col("data_hora_finalizacao_atendimento").cast("timestamp"))
        .withColumn("tempo_espera_atendimento_min", col("tempo_espera_atendimento_min").cast("double"))
        .withColumn("tempo_atendimento_min", col("tempo_atendimento_min").cast("double"))
        .withColumn("dia", col("dia").cast("int"))
        .withColumn("mes", col("mes").cast("int"))
        .withColumn("ano", col("ano").cast("int"))
        .withColumn("dia_semana_num", col("dia_semana_num").cast("int"))
        .withColumn("nome_dia", col("nome_dia").cast("string"))
        .withColumn("trimestre", col("trimestre").cast("int"))
        .withColumn("semana_do_ano", col("semana_do_ano").cast("int"))
        .withColumn("flag_fim_de_semana", col("flag_fim_de_semana").cast("boolean"))
        .withColumn("ciclo_operacional", col("ciclo_operacional").cast("string"))
        .withColumn("evento_sazonal", col("evento_sazonal").cast("string"))
        .withColumn("motivo", col("motivo").cast("string"))
        .withColumn("resolvido", col("resolvido").cast("boolean"))
        .withColumn("nome_canal", col("nome_canal").cast("string"))
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
        .withColumn("nota_atendimento", col("nota_atendimento").cast("int"))
        .withColumn("valor_custo", col("valor_custo").cast("double"))
)

df_silver_fato_chamados.write.mode("overwrite").saveAsTable("silver.fato_chamados")

display(df_silver_fato_chamados.limit(10))